In [1]:
using DataFramesMeta, BenchmarkTools

In [ ]:
@btime let 
  local s = 0
  for i=1:100_000
    s += rand(1:10) 
    s -= rand(1:10) 
  end
  s
end

  290.375 μs (0 allocations: 0 bytes)


1940

In [ ]:
@btime let 
  local s = 0.0
  for i=1:100_000
    s += rand(1.0:10.0) 
    s -= rand(1.0:10.0) 
  end
  s
end

  433.250 μs (0 allocations: 0 bytes)


161715.7761904773

In [231]:
@btime let 
  local s = big(0)
  for i=1:100_000
    s += rand(big(1):big(10)^100)
    s -= rand(big(1):big(10)^100)
  end
  s
end

  48.712 ms (2999942 allocations: 96.13 MiB)


1571331743465877238393046438061053633913351643499566500573915082518374696687626827976155968539429669568

In [ ]:
function calc_e(n::Integer)
  sum = big(1.0)
  for k=1:n
    sum += 1/factorial(k)
  end
  sum
end


calc_e (generic function with 1 method)

In [3]:
abs(calc_e(10)-exp(1))

2.7312660622705282748566048278693330075839185155928134918212890625e-8

In [ ]:
function num_digits_e(approx::T) where T <: Real
  local e_val = exp(one(T))
  approx == e_val ? floor(Int, log10(2)*precision(approx)) : floor(Int, -log10(abs(approx - e_val)))
end

num_digits_e (generic function with 1 method)

In [122]:
num_digits_e(calc_e(10))

7

In [123]:
num_digits_e(calc_e(20))

19

In [ ]:
function calc_e(n::Integer)
  sum = big(1.0)
  for k=1:n
    sum += 1/factorial(big(k))
  end
  sum
end

calc_e (generic function with 1 method)

In [125]:
num_digits_e(approx::Real) = floor(Int, -log10(abs(approx - exp(big(1.0)))))

num_digits_e (generic function with 1 method)

In [126]:
num_digits_e(calc_e(30))

33

In [127]:
calc_e(30)

2.718281828459045235360287471352662372225702130983481815815378576523792800791153

In [ ]:
@chain DataFrame(n=10:10:100) begin
  @transform(:digits = num_digits_e.(calc_e.(:n)))
end


Row,n,digits
,Int64,Int64
1,10,7
2,20,19
3,30,33
4,40,49
5,50,66
6,60,75
7,70,75
8,80,75
9,90,75


In [129]:
big(1.0)

1.0

In [130]:
precision(big(1.0))

256

In [ ]:
setprecision(1024) do 
  e1 = calc_e(100)
  num_digits_e(e1)
end

159

In [ ]:
@btime setprecision(2^12) do 
  e1 = calc_e(500)
  num_digits_e(e1)
end

  2.039 ms (7786 allocations: 1.06 MiB)


1136

In [ ]:
setprecision(2^12) do 
  @btime e1 = calc_e(500)
end

  1.764 ms (7730 allocations: 1.04 MiB)


2.71828182845904523536028747135266249775724709369995957496696762772407663035354759457138217852516642742746639193200305992181741359662904357290033429526059563073813232862794349076323382988075319525101901157383418793070215408914993488416750924476146066808226480016847741185374234544243710753907774499206955170276183860626133138458300075204493382656029760673711320070932870912744374704723069697720931014169283681902551510865746377211125238978442505695369677078544996996794686445490598793163688923009879312773617821542499922957635148220826989519366803318252886939849646510582093923982948879332036250944311730123819706841614039701983767932068328237646480429531180232878250981945581530175671736133206981125099618188159304169035159888851934580727386673858942287922849989208680582574927961048419844436346324496848756023362482704197862320900216099023530436994184914631409343173814364054625315209618369088870701676839642437814059271456354906130310720851038375051011574770417189861068739696552126715468895703503

In [170]:
@btime factorial(big(500))

  1.167 μs (8 allocations: 640 bytes)


1220136825991110068701238785423046926253574342803192842192413588385845373153881997605496447502203281863013616477148203584163378722078177200480785205159329285477907571939330603772960859086270429174547882424912726344305670173270769461062802310452644218878789465754777149863494367781037644274033827365397471386477878495438489595537537990423241061271326984327745715546309977202781014561081188373709531016356324432987029563896628911658974769572087926928871281780070265174507768410719624390394322536422605234945850129918571501248706961568141625359056693423813008856249246891564126775654481886506593847951775360894005745238940335798476363944905313062323749066445048824665075946735862074637925184200459369692981022263971952597190945217823331756934581508552332820762820023402626907898342451712006207714640979456116127629145951237229913340169552363850942885592018727433795173014586357570828355780158735432768888680120399882384702151467605445407663535984174430480128938313896881639487469658817504506926365338175

In [171]:
@btime 1/factorial(big(500))

  1.467 μs (10 allocations: 832 bytes)


8.195802132172396309504761462495484735303103973134542418553337729108316576457446e-1135

In [ ]:
function calc_e2(n::Integer)
  sum = big(1.0)
  fact = big(1)
  for k=1:n
    fact *= k
    sum += 1/fact
  end
  sum
end

calc_e2 (generic function with 1 method)

In [ ]:
setprecision(2^12) do 
  e1 = @btime calc_e2(500)
  num_digits_e(e1)
end

  1.452 ms (6005 allocations: 1.00 MiB)


1136

In [174]:
1.888/2.162

0.8732654949121184

In [ ]:
function calc_e3(n::Integer)
  sum = big(1.0)
  term = big(1.0)
  for k=1:n
    term /= big(k)
    sum += term
  end
  sum
end

calc_e3 (generic function with 1 method)

In [ ]:
setprecision(2^12) do 
  e1 = @btime calc_e3(500)
  num_digits_e(e1)
end

  305.333 μs (4006 allocations: 618.38 KiB)


1136

In [177]:
591/1242

0.4758454106280193

In [178]:
1-1//3+1//5-1//7

76//105

In [ ]:
arctan_terms(k) = k== 0 ? (P=1, Q=1, T=1) : (P=-1*(2k-1), Q= 2k+1, T=-1*(2k-1))

arctan_terms (generic function with 1 method)

In [180]:
[arctan_terms(k) for k=0:3]

4-element Vector{@NamedTuple{P::Int64, Q::Int64, T::Int64}}:
 (P = 1, Q = 1, T = 1)
 (P = -1, Q = 3, T = -1)
 (P = -3, Q = 5, T = -3)
 (P = -5, Q = 7, T = -5)

In [181]:
function binary_split(a::Integer, b::Integer, init::Function)
    if b - a == 1
        return init(a)
    else
        m = (a + b) ÷ 2
        left = binary_split(a, m, init)
        right = binary_split(m, b, init)
        return (P=left.P*right.P, Q=left.Q*right.Q, T=left.T * right.Q + left.P * right.T)
    end
end

binary_split (generic function with 2 methods)

In [ ]:
binary_split(0,1, arctan_terms), binary_split(1,2, arctan_terms), 
  binary_split(2,3, arctan_terms),binary_split(3,4, arctan_terms)

((P = 1, Q = 1, T = 1), (P = -1, Q = 3, T = -1), (P = -3, Q = 5, T = -3), (P = -5, Q = 7, T = -5))

In [ ]:
binary_split(0,2, arctan_terms), binary_split(2,4, arctan_terms)

((P = -1, Q = 3, T = 2), (P = 15, Q = 35, T = -6))

In [184]:
four_terms = binary_split(0, 4, arctan_terms)

(P = -15, Q = 105, T = 76)

In [185]:
four_terms.T/four_terms.Q, 4*four_terms.T/four_terms.Q

(0.7238095238095238, 2.895238095238095)

In [ ]:
exp_terms = arctan_terms(k) = k== 0 ? (P=1, Q=1, T=1) : (P=1, Q=k, T=1)

arctan_terms (generic function with 1 method)

In [187]:
[exp_terms(k) for k=0:3]

4-element Vector{@NamedTuple{P::Int64, Q::Int64, T::Int64}}:
 (P = 1, Q = 1, T = 1)
 (P = 1, Q = 1, T = 1)
 (P = 1, Q = 2, T = 1)
 (P = 1, Q = 3, T = 1)

In [ ]:
binary_split(0,2, exp_terms), binary_split(2,4, exp_terms)

((P = 1, Q = 1, T = 2), (P = 1, Q = 6, T = 4))

In [ ]:
binary_split(0,4,exp_terms)

(P = 1, Q = 6, T = 16)

In [ ]:
binary_split(0,64,exp_terms)

(P = 1, Q = 1585267068834414592, T = -2674445815446327044)

In [ ]:
@btime e64 = binary_split(big(0),big(64),exp_terms)

  57.834 μs (1455 allocations: 32.95 KiB)


(P = 1, Q = 1982608315404440064116146708361898137544773690227268628106279599612729753600000000000000, T = 5389288156715688797574170724637602641365768984616696724053991331785579414134712244547836)

In [195]:
e64.T/e64.Q

2.718281828459045235360287471352662497757247093699959574966967627724076630353555

In [193]:
e64.T/e64.Q - exp(big(1))

0.0

In [157]:
e64.T/e64.Q == exp(big(1))

true

In [158]:
floor(Int, log10(2)*precision(e64.T/e64.Q))

77

In [ ]:
num_digits_e(e64.T/e64.Q)

LoadError: InexactError: round(Int64, Inf)

In [ ]:
e128 = binary_split(big(0),big(128),exp_terms)

(P = 1, Q = 3012660018457659544809977077527059692324164918673621799053346900596667207618480809067860692097713761984609779945772783965563851033300772326297773087851869982500270661791244122597621760000000000000000000000000000000, T = 8189258983498547755301136726331018182795341082989472417800294638981943850974416517674504393522298820068493412557445147820427301212765815466137359501584796845232882984212354973723008236769314619218407567845294552380)

In [197]:
e128.T//e128.Q

851274322608996648160201322903432243533819239395995053825394453116626179934970531982796714503357465703585593820940244056177474138541145058850037370227109859171817358026232325750832457044627299295052761730280099//313166322084995794678791796000733855750952694248817234828830239147262703494644574747178866122423467981768168393531474424694787009698624981943635456117658002338905474198674025218048000000000000000000000000000000

In [198]:
e128.T/e128.Q

2.71828182845904523536028747135266249775724709369995957496696762772407663035352

In [215]:
setprecision(2^12) do
    e_approx = @btime let
        a = binary_split(big(0), big(128), exp_terms)
        a.T/a.Q
    end
    num_digits_e(e_approx)
end

  115.041 μs (2939 allocations: 68.55 KiB)


215

In [ ]:
num_digits_e(e128.T/e128.Q)

76

In [ ]:
@btime setprecision(2^12) do 
  e_approx = binary_split(big(0),big(2)^9,exp_terms)
  num_digits_e(e_approx.T/e_approx.Q)
end

  788.208 μs (11831 allocations: 295.73 KiB)


1166